[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S07/S07_02_ridge_lasso.ipynb)

# S07 · 02 — Regularización: Ridge y Lasso

**Tiempo estimado:** 50 min · **Sprint 7** · M3 (UD 3.3)

**Contexto TurisData.** TurisData abre un **alojamiento nuevo** y solo tiene 120 reservas
para estimar su tarifa. Con tan pocos datos y muchas variables (y sus combinaciones), un
modelo lineal "normal" se **sobreajusta**: memoriza el ruido. La solución: **regularizar**,
es decir, penalizar los coeficientes grandes para que el modelo se mantenga sencillo.

**Al terminar sabrás:**
1. Reconocer el sobreajuste de una regresión con muchas variables y pocos datos.
2. Explicar qué hacen **Ridge** (L2: encoge coeficientes) y **Lasso** (L1: los lleva a cero).
3. Elegir el parámetro de regularización `alpha` con validación cruzada.
4. Usar Lasso para **seleccionar variables**.

In [ ]:
# Preparación (ejecuta esta celda siempre la primera)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta


In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso, RidgeCV, LassoCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.base import clone

warnings.filterwarnings("ignore", category=UserWarning)
SEMILLA = 42
sns.set_theme(style="whitegrid")

df = pd.read_csv(dato("reservas_turisdata.csv"), parse_dates=["fecha_reserva", "fecha_llegada"])
df["mes_llegada"] = df["fecha_llegada"].dt.month
df["temporada_alta"] = df["mes_llegada"].isin([1, 2, 3, 7, 8, 12]).astype(int)
df["pais_origen"] = df["pais_origen"].fillna("desconocido")

columnas_num = ["antelacion_dias", "noches", "adultos", "ninos", "peticiones_especiales",
                "cliente_repetidor", "cancelaciones_previas", "mes_llegada", "temporada_alta"]
columnas_cat = ["tipo_habitacion", "regimen", "canal", "pais_origen", "tarifa", "deposito"]
X, y = df[columnas_num + columnas_cat], df["precio_noche"]

# Simulamos el alojamiento nuevo: solo 120 reservas para entrenar, el resto para evaluar
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=120, random_state=SEMILLA)
print("Reservas para entrenar:", len(X_train), "· para evaluar:", len(X_test))

prep = ColumnTransformer([("num", StandardScaler(), columnas_num),
                          ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), columnas_cat)])

## 1. El problema: muchas variables, pocos datos

Construimos un modelo "generoso": las variables originales **y todas sus combinaciones de
grado 2** (productos y cuadrados). Salen cientos de columnas para solo 120 filas.
Solo unas pocas variables influyen de verdad en el precio (tipo de habitación y temporada).

In [ ]:
def modelo_con(regresor):
    """Preprocesado + variables de grado 2 + escalado + regresor."""
    return make_pipeline(clone(prep), PolynomialFeatures(2, include_bias=False), StandardScaler(), regresor)

ols = modelo_con(LinearRegression()).fit(X_train, y_train)
print("Número de columnas del modelo:", ols[-1].coef_.shape[0])
print(f"R² en entrenamiento: {ols.score(X_train, y_train):.3f} · R² en prueba: {ols.score(X_test, y_test):.3f}")

**Ejercicio 1.** Guarda en `r2_train_ols` y `r2_test_ols` los R² de `ols` en train y test,
y en `brecha_ols` su diferencia (train − test). ¿Es una brecha sana?

In [ ]:
# TODO: ols.score(X_train, y_train) y ols.score(X_test, y_test)
...
assert r2_train_ols > 0.99, "En train el R² debería ser casi 1: el modelo memoriza"
assert brecha_ols > 0.2, "La brecha debería ser grande (sobreajuste)"

## 2. Ridge (L2): encoger los coeficientes

**Idea.** Ridge minimiza el error **más** una penalización `alpha · Σ coeficiente²`.
- `alpha` pequeño → casi una regresión normal.
- `alpha` grande → todos los coeficientes se encogen hacia 0 (modelo más simple, más sesgo).

Ridge **reparte** el peso entre variables parecidas; rara vez deja un coeficiente exactamente en 0.
**Es imprescindible escalar** las variables antes, para que la penalización sea justa.

Vemos el efecto sobre un modelo sencillo (sin polinomios) al variar `alpha`.

In [ ]:
Xp_train = prep.fit_transform(X_train)
nombres = prep.get_feature_names_out()
alphas = np.logspace(-2, 3, 30)
trayectoria_ridge = np.array([Ridge(alpha=a).fit(Xp_train, y_train).coef_ for a in alphas])
trayectoria_lasso = np.array([Lasso(alpha=a, max_iter=50000).fit(Xp_train, y_train).coef_ for a in alphas])

fig, ejes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
for ax, tray, titulo in [(ejes[0], trayectoria_ridge, "Ridge (L2): los coeficientes se encogen"),
                         (ejes[1], trayectoria_lasso, "Lasso (L1): los coeficientes llegan a cero")]:
    ax.plot(alphas, tray, lw=1)
    ax.set_xscale("log")
    ax.set(title=titulo, xlabel="alpha (escala logarítmica)")
    ax.axhline(0, color="black", lw=0.8)
ejes[0].set_ylabel("Valor del coeficiente (€)")
plt.tight_layout()
plt.show()

## 3. Elegir `alpha` con validación cruzada

`RidgeCV` y `LassoCV` prueban una lista de valores de `alpha` con validación cruzada
**dentro de train** y se quedan con el mejor. El test **no se toca** hasta el final.

**Ejercicio 2.** Crea `ridge` con `modelo_con(RidgeCV(alphas=np.logspace(-1, 4, 30)))` y ajústalo
con train. Guarda en `alpha_ridge` el alpha elegido (`ridge[-1].alpha_`) y en `r2_test_ridge`
su R² en test.

In [ ]:
# TODO: modelo_con(RidgeCV(...)).fit(X_train, y_train); el alpha elegido está en ridge[-1].alpha_
...
assert alpha_ridge > 0.1, "RidgeCV debe haber elegido un alpha > 0,1"
assert r2_test_ridge > r2_test_ols, "Ridge debería mejorar al modelo sin regularizar en test"
print(f"Ridge: alpha = {alpha_ridge:.1f} · R² test = {r2_test_ridge:.3f}")

## 4. Lasso (L1): seleccionar variables

**Idea.** Lasso penaliza `alpha · Σ |coeficiente|`. Esta penalización tiene un efecto curioso:
**lleva a cero** los coeficientes de las variables poco útiles. El resultado es un modelo
**disperso** (*sparse*) que además hace **selección de variables**. Es ideal cuando sospechas
que solo unas pocas variables importan.

**Ejercicio 3.** Crea `lasso` con `modelo_con(LassoCV(cv=5, max_iter=50000, random_state=SEMILLA))`,
ajústalo con train y guarda en `r2_test_lasso` su R² en test y en `n_no_cero` cuántos
coeficientes son distintos de cero (`np.sum(lasso[-1].coef_ != 0)`).

In [ ]:
# TODO: el LassoCV es el último paso del pipeline: lasso[-1].coef_
...
assert r2_test_lasso > 0.8, "Lasso debería superar 0,8 de R² en test"
assert n_no_cero < 80, "Lasso debería quedarse con muy pocos coeficientes de los cientos posibles"
print(f"Lasso: R² test = {r2_test_lasso:.3f} · coeficientes distintos de cero: {n_no_cero} de {lasso[-1].coef_.shape[0]}")

In [ ]:
# ¿Qué variables ha conservado Lasso? (las de mayor peso)
nombres_poly = lasso[:-1].get_feature_names_out()
coef_lasso = pd.Series(lasso[-1].coef_, index=nombres_poly)
principales = coef_lasso[coef_lasso != 0].sort_values(key=np.abs, ascending=False).head(8)
principales.round(2)

## 5. Comparativa final

**Idea.** Ninguno gana siempre:
- **Ridge** es mejor cuando **muchas** variables aportan un poco.
- **Lasso** es mejor cuando **pocas** variables lo explican casi todo (nuestro caso) y además da un modelo fácil de explicar.
- **ElasticNet** mezcla ambas penalizaciones (sirve cuando hay variables muy correlacionadas).

In [ ]:
comparativa = pd.DataFrame({
    "R² train": [ols.score(X_train, y_train), ridge.score(X_train, y_train), lasso.score(X_train, y_train)],
    "R² test": [r2_test_ols, r2_test_ridge, r2_test_lasso],
}, index=["Sin regularizar", "Ridge", "Lasso"]).round(3)
comparativa

## 6. Mini-reto: ¿cuántos datos hacen falta?

Repite el experimento con distintos tamaños de entrenamiento (`tamanos = [60, 120, 300, 1000]`)
y guarda en la tabla `curva` el R² en test de la regresión sin regularizar y de Lasso
(columnas `"ols"` y `"lasso"`, una fila por tamaño). Usa siempre las mismas 1000 reservas
de evaluación (`X_eval`, `y_eval`) para comparar de forma justa.

In [ ]:
tamanos = [60, 120, 300, 1000]
X_res, X_eval, y_res, y_eval = train_test_split(X, y, test_size=1000, random_state=SEMILLA)
# TODO: para cada tamaño toma X_res.iloc[:n], ajusta los dos modelos y guarda X_eval en el score
...
assert list(curva.index) == tamanos and list(curva.columns) == ["ols", "lasso"], "Índice = tamaños; columnas 'ols' y 'lasso'"
assert curva.loc[60, "lasso"] > curva.loc[60, "ols"], "Con pocos datos, Lasso debería superar al modelo sin regularizar"
curva

## 7. Para reflexionar

1. ¿Por qué con 1000 reservas la ventaja de regularizar se reduce?
2. Ridge no dejó ningún coeficiente en cero, pero Lasso sí. ¿Qué preferirías si el cliente pide "las 5 cosas que más influyen en el precio"?
3. ¿Qué habría pasado si no hubiéramos escalado las variables?
4. ¿Por qué `alpha` se elige con validación cruzada en train y no mirando el test?

## 8. Resumen: qué has aprendido
- Muchas variables + pocos datos = sobreajuste (train perfecto, test mediocre).
- Ridge encoge coeficientes; Lasso además los anula (selección de variables).
- `alpha` se elige con `RidgeCV`/`LassoCV`; siempre con las variables escaladas.
- Con más datos, la regularización pesa menos.